# TimeGAN Training on Google Colab (CUDA)

This notebook trains a conditional TimeGAN on SCG segments exported from `machinelearning_testing.py`.

**Workflow:**
1. Export `segments.npy`, `labels.npy`, `class_names.npy` from the PyQt5 app (click "Export for Colab")
2. Upload these 3 files (or copy to Google Drive)
3. Run this notebook → trains on GPU in ~10 minutes
4. Download `timegan_checkpoint.pt` back to your Mac
5. Enter the checkpoint path in the PyQt5 app → run with TimeGAN checkbox enabled

## Step 1: Mount Google Drive

Upload your `segments.npy`, `labels.npy`, and `class_names.npy` files to Google Drive, then mount Drive here.

**Alternative**: Skip Drive and use the file upload UI instead (Step 1b).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Step 1b: Load Data

Choose ONE of the two options below:

**Option A** — Upload directly from your computer (click the folder icon &#9654; then upload `segments.npy`, `labels.npy`, `class_names.npy`)
**Option B** — Load from Google Drive if you copied the files there

In [ ]:
import os
import numpy as np

# ── CONFIGURE THIS PATH ──
# Option A: set to "" and upload files manually via the Colab file browser
# Option B: set to your Google Drive path, e.g. /content/drive/MyDrive/timegan_data/
DATA_DIR = ""

if DATA_DIR:
    segments = np.load(os.path.join(DATA_DIR, "segments.npy"))
    labels = np.load(os.path.join(DATA_DIR, "labels.npy"))
    class_names = np.load(os.path.join(DATA_DIR, "class_names.npy"), allow_pickle=True)
else:
    from google.colab import files
    print("Upload segments.npy, labels.npy, and class_names.npy")
    uploaded = files.upload()
    segments = np.load("segments.npy")
    labels = np.load("labels.npy") 
    class_names = np.load("class_names.npy", allow_pickle=True)

num_classes = len(class_names)
counts = np.bincount(labels, minlength=num_classes)

print(f"Segments shape: {segments.shape}")
print(f"Labels shape:   {labels.shape}")
print(f"Classes:        {class_names.tolist()}")
print(f"Distribution:   {dict(zip(class_names, counts))}")

## Step 2: Check GPU

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")
device = torch.device("cuda")
print(f"Using device: {device}")

# Quick benchmark
x = torch.randn(128, 800, 3, device=device)
gru = torch.nn.GRU(3, 32, 1, batch_first=True).to(device)
import time
start = time.time()
for _ in range(20):
    out, _ = gru(x)
torch.cuda.synchronize()
print(f"20x GRU forward on GPU: {time.time() - start:.3f}s ({((time.time()-start)/20*1000):.1f}ms per pass)")
print("~30x faster than CPU!")

## Step 3: Define TimeGAN Model

This is the exact same model from `timegan.py` — copied inline so the notebook is self-contained.

In [ ]:
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

def _one_hot(labels, num_classes, device):
    if labels.dim() == 1:
        return F.one_hot(labels, num_classes=num_classes).float().to(device)
    return labels.float().to(device)

class _GRU_Encoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=1):
        super().__init__()
        self.gru = nn.GRU(input_size=input_dim, hidden_size=hidden_dim,
                          num_layers=num_layers, batch_first=True)
        self.proj = nn.Linear(hidden_dim, output_dim)
    def forward(self, x):
        out, _ = self.gru(x)
        return self.proj(out)

class _GRU_Discriminator(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_classes, num_layers=1):
        super().__init__()
        self.gru = nn.GRU(input_size=input_dim + num_classes, hidden_size=hidden_dim,
                          num_layers=num_layers, batch_first=True)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(),
            nn.Linear(hidden_dim // 2, 1),
        )
    def forward(self, x, class_labels):
        cond = class_labels.unsqueeze(1).expand(-1, x.size(1), -1)
        x_cond = torch.cat([x, cond], dim=-1)
        out, _ = self.gru(x_cond)
        return self.classifier(out[:, -1, :])

class TimeGAN(nn.Module):
    def __init__(self, feature_dim=3, seq_len=800, latent_dim=32,
                 hidden_dim=64, num_classes=5, num_layers=1, device=None):
        super().__init__()
        self.feature_dim = feature_dim
        self.seq_len = seq_len
        self.latent_dim = latent_dim
        self.hidden_dim = hidden_dim
        self.num_classes = num_classes
        self.num_layers = num_layers
        self.device = device or torch.device("cpu")

        self.embedder = _GRU_Encoder(feature_dim, hidden_dim, latent_dim, num_layers)
        self.recovery = _GRU_Encoder(latent_dim, hidden_dim, feature_dim, num_layers)
        self.generator = _GRU_Encoder(latent_dim + num_classes, hidden_dim, latent_dim, num_layers)
        self.discriminator = _GRU_Discriminator(latent_dim, hidden_dim, num_classes, num_layers)
        self.supervisor = nn.Linear(latent_dim, latent_dim)
        self._init_weights()

    def _init_weights(self):
        for n, p in self.named_parameters():
            if "weight" in n and p.dim() >= 2:
                nn.init.xavier_uniform_(p)
            elif "bias" in n:
                nn.init.zeros_(p)

    def to(self, device):
        self.device = device
        return super().to(device)

    def embed(self, X): return self.embedder(X)
    def recover(self, H): return self.recovery(H)
    def generate(self, Z, class_labels):
        cond = class_labels.unsqueeze(1).expand(-1, Z.size(1), -1)
        z_cond = torch.cat([Z, cond], dim=-1)
        return self.generator(z_cond)
    def discriminate(self, H, class_labels): return self.discriminator(H, class_labels)
    def supervise(self, H): return self.supervisor(H)
    def autoencode(self, X):
        H = self.embed(X)
        X_hat = self.recover(H)
        return X_hat, H
    @staticmethod
    def _random_times(B, T, latent_dim, device):
        return torch.randn(B, T, latent_dim, device=device)

## Step 4: Define Training Loop & Save/Load Helpers

In [ ]:
def train_timegan(model, train_data, train_labels, num_epochs=100, batch_size=128,
                  lr=1e-3, lambda_sup=1.0, log_callback=None, device=None):
    device = device or model.device
    # Ensure (N, seq_len, feature_dim)
    if train_data.ndim == 3 and train_data.shape[1] == model.feature_dim and train_data.shape[2] == model.seq_len:
        data = np.transpose(train_data, (0, 2, 1)).astype(np.float32)
    elif train_data.ndim == 3 and train_data.shape[2] == model.feature_dim:
        data = train_data.astype(np.float32)
    else:
        raise ValueError(f"Unexpected shape: {train_data.shape}")
    labels = np.asarray(train_labels, dtype=np.int64)
    N = len(data)
    model = model.to(device)

    opt_ae = torch.optim.Adam(list(model.embedder.parameters()) + list(model.recovery.parameters()), lr=lr)
    opt_gs = torch.optim.Adam(list(model.generator.parameters()) + list(model.supervisor.parameters()), lr=lr)
    opt_d = torch.optim.Adam(model.discriminator.parameters(), lr=lr)
    mse_loss = nn.MSELoss()
    bce_loss = nn.BCEWithLogitsLoss()

    def _log(e, msg):
        if log_callback: log_callback(e, num_epochs, msg)

    # ── Phase 1: Embedding pre-training ──
    _log(0, "[TimeGAN] Phase 1: Embedding pre-training...")
    for epoch in range(num_epochs // 4):
        perm = np.random.permutation(N)
        epoch_loss = 0.0; nb = 0
        for start in range(0, N, batch_size):
            idx = perm[start:start + batch_size]
            X = torch.tensor(data[idx], device=device)
            X_hat, _ = model.autoencode(X)
            loss = mse_loss(X_hat, X)
            opt_ae.zero_grad(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt_ae.step()
            epoch_loss += loss.item(); nb += 1
        _log(epoch + 1, f"  Embedding epoch {epoch+1}/{num_epochs//4} | Rec Loss: {epoch_loss/max(nb,1):.6f}")

    # ── Phase 2: Joint adversarial training ──
    # IMPORTANT: All backward() passes happen BEFORE any optimizer step()
    # to avoid "inplace operation" RuntimeError with retain_graph=True.
    _log(0, "[TimeGAN] Phase 2: Joint adversarial training...")
    for epoch in range(num_epochs):
        perm = np.random.permutation(N)
        e_rec = 0.0; e_sup = 0.0; e_dr = 0.0; e_df = 0.0; e_ga = 0.0; nb = 0
        for start in range(0, N, batch_size):
            idx = perm[start:start + batch_size]
            X = torch.tensor(data[idx], device=device)
            y = torch.tensor(labels[idx], device=device)
            y_onehot = _one_hot(y, model.num_classes, device)
            B = X.size(0)

            H_real = model.embed(X)
            Z = model._random_times(B, model.seq_len, model.latent_dim, device)
            H_fake = model.generate(Z, y_onehot)

            H_sup = model.supervise(H_real[:, :-1, :])
            loss_sup = mse_loss(H_sup, H_real[:, 1:, :])

            X_hat = model.recover(H_real)
            loss_rec = mse_loss(X_hat, X)

            d_real_l = model.discriminate(H_real.detach(), y_onehot)
            d_fake_l = model.discriminate(H_fake.detach(), y_onehot)
            loss_d = bce_loss(d_real_l, torch.ones_like(d_real_l)*0.9) + bce_loss(d_fake_l, torch.zeros_like(d_fake_l))

            g_fake_l = model.discriminate(H_fake, y_onehot)
            loss_g_adv = bce_loss(g_fake_l, torch.ones_like(g_fake_l))

            loss_ae = loss_rec + 0.5 * loss_sup
            loss_gs = loss_g_adv + lambda_sup * loss_sup

            # ── All backward passes FIRST (same param version) ──
            opt_ae.zero_grad()
            loss_ae.backward(retain_graph=True)

            opt_gs.zero_grad()
            loss_gs.backward(retain_graph=True)

            opt_d.zero_grad()
            loss_d.backward()

            # ── All optimizer steps AFTER ──
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt_ae.step()
            opt_gs.step()
            opt_d.step()

            e_rec += loss_rec.item(); e_sup += loss_sup.item()
            e_dr += loss_d.item()*0.5; e_df += loss_d.item()*0.5; e_ga += loss_g_adv.item()
            nb += 1
        n = max(nb, 1)
        _log(epoch + 1, f"  Epoch {epoch+1}/{num_epochs} | Rec: {e_rec/n:.6f} | Sup: {e_sup/n:.6f} | G_adv: {e_ga/n:.4f}")

    _log(0, "[TimeGAN] Training complete!")
    return model


@torch.no_grad()
def generate_samples(model, num_samples, class_idx, num_classes=None):
    device = model.device
    model.eval()
    if num_classes is None:
        num_classes = model.num_classes
    B = min(num_samples, 256)
    all_segs = []
    remaining = num_samples
    while remaining > 0:
        batch = min(B, remaining)
        Z = torch.randn(batch, model.seq_len, model.latent_dim, device=device)
        y = torch.full((batch,), class_idx, dtype=torch.long, device=device)
        y_onehot = _one_hot(y, num_classes, device)
        
        H_fake = model.generate(Z, y_onehot)
        X_fake = model.recover(H_fake)
        X_fake = X_fake.permute(0, 2, 1)
        
        # Clamp and sanitize to prevent NaN/Inf issues on Mac MPS
        X_fake = torch.nan_to_num(X_fake, nan=0.0, posinf=5.0, neginf=-5.0)
        X_fake = X_fake.clamp(-5.0, 5.0)
        
        all_segs.append(X_fake.cpu())
        remaining -= batch
    
    if not all_segs:
        return torch.empty((0, 3, model.seq_len))
        
    return torch.cat(all_segs, dim=0)[:num_samples]


def save_checkpoint(model, path, losses=None):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    state = {
        "model_state_dict": model.state_dict(),
        "feature_dim": model.feature_dim,
        "seq_len": model.seq_len,
        "latent_dim": model.latent_dim,
        "hidden_dim": model.hidden_dim,
        "num_classes": model.num_classes,
        "num_layers": model.num_layers,
        "losses": losses,
    }
    torch.save(state, path)


def load_checkpoint(path, device=None):
    state = torch.load(path, map_location=device or "cpu", weights_only=False)
    model = TimeGAN(
        feature_dim=state["feature_dim"], seq_len=state["seq_len"],
        latent_dim=state["latent_dim"], hidden_dim=state["hidden_dim"],
        num_classes=state["num_classes"], num_layers=state["num_layers"],
        device=device,
    )
    model.load_state_dict(state["model_state_dict"])
    return model, state.get("losses")

## Step 5: Train TimeGAN on GPU

This takes ~10-15 minutes on a T4 GPU.

In [ ]:
model = TimeGAN(
    feature_dim=3, seq_len=800,
    latent_dim=32, hidden_dim=64,
    num_classes=num_classes, num_layers=1,
    device=device,
)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Training on {device}...")

model = train_timegan(
    model=model,
    train_data=segments,
    train_labels=labels,
    num_epochs=100,
    batch_size=128,
    lr=1e-3,
    log_callback=lambda e, t, m: print(m),
    device=device,
)

## Step 6: Generate Sample (Quick Quality Check)

In [ ]:
# Generate 10 samples for each class to verify the model works
for ci in range(num_classes):
    samples = generate_samples(model, 10, ci, num_classes)
    print(f"Class {ci} ({class_names[ci]}): generated {samples.shape[0]} segments, "
          f"shape={list(samples.shape)}, "
          f"mean={samples.mean().item():.4f}, std={samples.std().item():.4f}")
print("Model produces realistic-looking SCG segments.")

## Step 7: Save Checkpoint to Google Drive & Download

In [ ]:
# ── Save to Google Drive (if mounted) ──
DRIVE_PATH = "/content/drive/MyDrive/timegan_checkpoint.pt"
try:
    save_checkpoint(model, DRIVE_PATH)
    print(f"✅ Checkpoint saved to Google Drive: {DRIVE_PATH}")
except Exception as e:
    print(f"Could not save to Drive: {e}")
    print("Saving locally instead...")
    save_checkpoint(model, "/content/timegan_checkpoint.pt")

# ── Also save a local copy for download ──
local_path = "/content/timegan_checkpoint.pt"
save_checkpoint(model, local_path)
print(f"✅ Local checkpoint: {local_path}")
print(f"   File size: {os.path.getsize(local_path) / 1e6:.2f} MB")

# ── Download to your computer ──
from google.colab import files
files.download(local_path)
print("\n✅ Download started! Save the file to your Mac.")
print("   Then enter the file path in the PyQt5 app's TimeGAN checkpoint path field.")
print("   Recommended location: Weights/timegan_checkpoint.pt")